In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": { "parameterName": "LakehouseName", "defaultValue": "AzureCostAnalyzer_LH" },
    "id": { "parameterName": "LakehouseId", "defaultValue": "xxxxxxxx-xxxx-xxxx-xxxx-xxxxxxxxxxxx" },
    "workspaceId": { "parameterName": "WorkspaceId", "defaultValue": "yyyyyyyy-yyyy-yyyy-yyyy-yyyyyyyyyyyy" }
  }
}

# 07 · Gold · Anomalies

Builds **`gold_cost_anomalies`** — statistical anomaly detection on daily costs.

**Method**: Rolling 7-day mean + std dev → Z-score → flag if |Z| > 3

**Output**: Date × SubAccountName × ServiceName → EffectiveCost, RollingMean, StdDev, ZScore, IsAnomaly, Severity

**Dependencies**: `gold_cost_summary_daily`.

In [ ]:
# Configure default lakehouse using parameters
# mssparkutils is available in Fabric runtime (no import needed)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Window: 7-day rolling by SubAccountName + ServiceName
w = Window.partitionBy("SubAccountName", "ServiceName").orderBy("Date").rowsBetween(-6, 0)

anomalies_df = (
    spark.table("gold_cost_summary_daily")
         .withColumn("RollingMean", F.avg("EffectiveCost").over(w))
         .withColumn("RollingStdDev", F.stddev("EffectiveCost").over(w))
         .withColumn("ZScore", 
             F.when(F.col("RollingStdDev") > 0, 
                    (F.col("EffectiveCost") - F.col("RollingMean")) / F.col("RollingStdDev")
             ).otherwise(0)
         )
         .withColumn("IsAnomaly", F.when(F.abs(F.col("ZScore")) > 3, True).otherwise(False))
         .withColumn("Severity",
             F.when(F.abs(F.col("ZScore")) > 5, "Critical")
              .when(F.abs(F.col("ZScore")) > 3, "High")
              .otherwise("Normal")
         )
         .select("Date", "SubAccountName", "ServiceCategory", "ServiceName", "RegionName",
                 "EffectiveCost", "RollingMean", "RollingStdDev", "ZScore", "IsAnomaly", "Severity")
)

anomalies_df.write.format("delta").mode("overwrite").saveAsTable("gold_cost_anomalies")

anomaly_count = anomalies_df.filter(F.col("IsAnomaly") == True).count()
print(f"✓ Created gold_cost_anomalies with {anomalies_df.count():,} rows ({anomaly_count:,} anomalies detected)")